# 01 - Single hopfion: construct, verify topology, relax

Build an analytic Hopf-fibration ansatz, verify its Hopf invariant via the FFT method, then relax under damped LLG with DMI + anisotropy strong enough to stabilize Q_H=1 as a *metastable* minimum (we don't expect the global energy minimum to be the hopfion -- that's the uniform state; we want the hopfion to be a local minimum the dynamics can't escape).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from hopfion.grid import Grid
from hopfion.field import hopfion
from hopfion.topology import hopf_index, skyrmion_density_xy
from hopfion.energy import EnergyParams, total_energy
from hopfion.llg import relax
from hopfion.viz import slice_quiver, preimage_scatter

In [ ]:
# 48^3 cube with dx=0.3, hopfion size R=1.5 -- ~5 grid cells per hopfion radius.
g = Grid(48, 48, 48, 0.3, 0.3, 0.3, 'periodic')
m = hopfion(g, R=1.5, p=1, q=1)
Q0 = hopf_index(m, g)
print(f'Initial Hopf index Q_H = {Q0:+.4f}  (expected 1)')

In [ ]:
# Skyrmion charge per z-slice -- a Q_H=1 hopfion has opposite-sign charge on top/bottom that sums to zero.
Q_per_z = skyrmion_density_xy(m, g)
plt.plot(np.arange(g.nz)*g.dz - g.nz*g.dz/2, np.asarray(Q_per_z))
plt.xlabel('z'); plt.ylabel('Q_skyrmion per slice'); plt.grid(True)
plt.title('Skyrmion charge per (x, y) slice through the hopfion');

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
slice_quiver(m, g, plane='xy', stride=2, ax=axes[0]); axes[0].set_title('xy slice (z=0)')
slice_quiver(m, g, plane='xz', stride=2, ax=axes[1]); axes[1].set_title('xz slice (y=0)')

In [ ]:
# Linked-loop preimage signature: the +x and -x preimages link once for Q_H=1.
fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection='3d')
preimage_scatter(m, g, tol=0.1, ax=ax)
ax.set_title('Preimage scatter: +x (red) and -x (blue) loops link once');

In [ ]:
# Relax with parameters where Q_H=1 is metastable: high D, K balances exchange so the hopfion
# remains a local min. Found by parameter scan: D=1.5, K=0.7, A=1.
ep = EnergyParams(A_ex=1.0, D=1.5, Ku=0.7, easy_axis=(0, 0, 1), H_ext=(0, 0, 0.0))
print(f'Initial E = {total_energy(m, g, ep):.4f}')
m_relaxed = relax(m, g, ep, n_steps=300, dt=0.002)
print(f'Final   E = {total_energy(m_relaxed, g, ep):.4f}')
print(f'Final   Q_H = {hopf_index(m_relaxed, g):+.4f}  -- should still be 1 if the parameters stabilize the hopfion')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
slice_quiver(m, g, plane='xy', stride=2, ax=axes[0]); axes[0].set_title('ansatz, xy')
slice_quiver(m_relaxed, g, plane='xy', stride=2, ax=axes[1]); axes[1].set_title('relaxed, xy')